In [ ]:
!pip install -q transformers datasets evaluate jiwer librosa soundfile accelerate

In [ ]:
import os
os.makedirs("asr_assignment", exist_ok=True)

In [ ]:
import sys
sys.path.append('/content')

In [ ]:
!pip install -U datasets

In [ ]:
!pip uninstall -y datasets
!pip install datasets==2.18.0

In [ ]:
from datasets import load_dataset

dataset = load_dataset("google/fleurs", "hi_in")

In [ ]:
train_ds = dataset["train"].select(range(200))
test_ds = dataset["test"].select(range(50))

In [ ]:
from datasets import Audio

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
test_ds = test_ds.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

# ✅ Force Hindi decoding
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="hindi",
    task="transcribe"
)

# ✅ FIX: use generation_config instead
model.generation_config.suppress_tokens = []

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    inputs = processor(
        audio["array"],
        sampling_rate=16000,
        return_tensors="pt"
    )

    batch["input_features"] = inputs.input_features[0]

    batch["labels"] = processor.tokenizer(
        batch["transcription"],
        return_tensors="pt",
        padding=True
    ).input_ids[0]

    return batch

In [ ]:
train_ds = train_ds.map(prepare_dataset, remove_columns=train_ds.column_names)
test_ds = test_ds.map(prepare_dataset, remove_columns=test_ds.column_names)

In [ ]:
print(train_ds[0])

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
from tqdm import tqdm

def generate_predictions(dataset):
    preds = []
    refs = []

    for sample in tqdm(dataset):
        input_features = torch.tensor(sample["input_features"]).unsqueeze(0).to(device)

        with torch.no_grad():
            predicted_ids = model.generate(
    input_features,
    max_length=225,
    num_beams=1
)

        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        preds.append(transcription)
        refs.append(processor.tokenizer.decode(sample["labels"], skip_special_tokens=True))

    return preds, refs

In [ ]:
baseline_preds, refs = generate_predictions(test_ds)

In [ ]:
import re

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[।!?.,]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def keep_hindi_only(text):
    return re.sub(r"[^\u0900-\u097F\s]", "", text)

In [ ]:
cleaned_preds = [
    normalize_text(keep_hindi_only(p))
    for p in baseline_preds
]

cleaned_refs = [
    normalize_text(keep_hindi_only(r))
    for r in refs
]

In [ ]:
for i in range(5):
    print("REF:", refs[i])
    print("PRED:", baseline_preds[i])
    print("-"*50)

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")

cleaned_wer = wer_metric.compute(
    predictions=cleaned_preds,
    references=cleaned_refs
)

print("Final Cleaned WER:", cleaned_wer)

In [ ]:
from asr_assignment.error_analysis import ErrorTaxonomyBuilder

rows = []
for i in range(len(refs)):
    rows.append({
        "sample_id": str(i),
        "reference": refs[i],
        "hypothesis": baseline_preds[i]
    })

builder = ErrorTaxonomyBuilder()
cases = builder.build_cases(rows)

print(len(cases))  # should be many errors

In [ ]:
from asr_assignment.error_analysis import systematic_sample

sampled_cases = systematic_sample(cases, sample_size=25)

for case in sampled_cases[:5]:
    print(case)

In [ ]:
from collections import Counter

error_counts = Counter(case.error_type for case in sampled_cases)
print(error_counts)

In [ ]:
for case in sampled_cases[:10]:
    print("REF:", case.reference)
    print("PRED:", case.hypothesis)
    print("TYPE:", case.error_type)
    print("-"*50)

In [ ]:
import re

def is_urdu(text):
    return bool(re.search(r"[\u0600-\u06FF]", text))

def is_devanagari(text):
    return bool(re.search(r"[\u0900-\u097F]", text))

In [ ]:
def classify_advanced(case):
    ref = case.reference
    hyp = case.hypothesis

    # 1. Script mismatch
    if is_urdu(hyp) and is_devanagari(ref):
        return "script-mismatch"

    # 2. English borrowing (already handled)
    if case.error_type == "english-borrowing":
        return "english-borrowing"

    # 3. Deletion / Insertion
    if case.error_type == "deletion":
        return "deletion"
    if case.error_type == "insertion":
        return "insertion"

    # 4. Everything else → phonetic / substitution
    return "phonetic-error"

In [ ]:
enhanced_labels = []

for case in sampled_cases:
    label = classify_advanced(case)
    enhanced_labels.append(label)

from collections import Counter
print(Counter(enhanced_labels))

In [ ]:
for case, label in zip(sampled_cases, enhanced_labels):
    if label == "script-mismatch":
        print("REF:", case.reference)
        print("PRED:", case.hypothesis)
        print()

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")

baseline_wer = wer_metric.compute(
    predictions=baseline_preds,
    references=refs
)

print("Baseline WER:", baseline_wer)

In [ ]:
print("Before WER:", baseline_wer)
print("After cleanup WER:", cleaned_wer)

In [ ]:
def extract_words(texts):
    words = set()
    for text in texts:
        for w in text.split():
            words.add(w)
    return list(words)

all_words = extract_words(refs)
print("Total unique words:", len(all_words))

In [ ]:
from collections import Counter

word_freq = Counter()

for text in refs:
    for w in text.split():
        word_freq[w] += 1

In [ ]:
import re

def is_hindi(word):
    return bool(re.match(r'^[\u0900-\u097F]+$', word))

def classify_word(word):
    freq = word_freq[word]

    # High confidence correct
    if freq > 5 and is_hindi(word):
        return "correct", "high"

    # Medium
    if freq > 2:
        return "correct", "medium"

    # Low confidence (possible error)
    return "incorrect", "low"

In [ ]:
results = []

for word in all_words:
    label, confidence = classify_word(word)
    results.append((word, label, confidence))

len(results)

In [ ]:
low_conf = [r for r in results if r[2] == "low"]

print("Low confidence words:", len(low_conf))

for w in low_conf[:20]:
    print(w)